In [ ]:
import json

def convert_to_forms_api_requests(survey_data):
    """
    Converts custom survey JSON into Google Forms API batchUpdate requests.
    """
    requests = []
    current_section = None
    index = 0
    
    for item in survey_data:
        # 1. Handle Section Breaks
        # If the section_id changes, insert a PAGE_BREAK item
        if item["section_id"] != current_section:
            if current_section is not None: 
                requests.append({
                    "createItem": {
                        "item": {
                            "title": item["section_id"].replace("_", " ").title(),
                            "pageBreakItem": {}
                        },
                        "location": {"index": index}
                    }
                })
                index += 1
            current_section = item["section_id"]
            
        # 2. Build the Question Item
        q_type = item["type"]
        form_item = {
            "title": item["question"]
        }
        
        if q_type == "single_choice":
            form_item["questionItem"] = {
                "question": {
                    "choiceQuestion": {
                        "type": "RADIO",
                        "options": [{"value": opt} for opt in item["options"]]
                    }
                }
            }
        elif q_type == "multiple_choice":
            form_item["questionItem"] = {
                "question": {
                    "choiceQuestion": {
                        "type": "CHECKBOX",
                        "options": [{"value": opt} for opt in item["options"]]
                    }
                }
            }
        elif q_type == "text":
            form_item["questionItem"] = {
                "question": {
                    "textQuestion": {
                        "paragraph": True # True for long answers, False for short answers
                    }
                }
            }
            
        # 3. Append the Question Request
        requests.append({
            "createItem": {
                "item": form_item,
                "location": {"index": index}
            }
        })
        index += 1
        
    return {"requests": requests}

# Example Usage:
if __name__ == "__main__":
    # Assuming 'survey.json' contains your provided data array
    with open('./results/final_survey.json', 'r', encoding='utf-8') as f:
        local_survey_data = json.load(f)
        
    api_payload = convert_to_forms_api_requests(local_survey_data)
    
    # This payload can now be passed directly to the forms().batchUpdate() method
    print(json.dumps(api_payload, indent=2, ensure_ascii=False))

In [5]:
import json
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

def mcp_survey_executor(questions_data: str,) -> str:
    configs = json.load(open('./configs.json'))["GOOGLE_KEYS"]
    creds = None
    # CHECK FOR EXISTING CREDENTIALS
    if os.path.exists(configs["Token_json"]):
        creds = Credentials.from_authorized_user_file(configs["Token_json"], configs['SCOPES'])
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # Ensure the path to your client secrets JSON is correct
            flow = InstalledAppFlow.from_client_secrets_file(
                configs["GOOGLE_APPLICATION_CREDENTIALS"], configs['SCOPES']
            )
            # This replaces run_flow and works correctly in Jupyter
            creds = flow.run_local_server(port=0) 
        with open(configs["Token_json"], "w") as token:
            token.write(creds.to_json())
    
    #BUILD SERVICE
    form_service = build("forms", "v1", credentials=creds)
    # 3. DEFINE FORM DATA
    result = form_service.forms().create(body={
        "info": {
            "title": "Quickstart Form",
        }
    }).execute()
    form_id = result["formId"]
    print(f"Form Created! ID: {form_id}")

    api_payload = convert_to_forms_api_requests(questions_data)

    form_service.forms().batchUpdate(formId=form_id, body=api_payload).execute()

    # Verify and print the final form structure
    final_form = form_service.forms().get(formId=form_id).execute()
    

mcp_survey_executor(json.load(open('./results/final_survey.json', 'r', encoding='utf-8')))

Form Created! ID: 1_kNRYRdoTmXduAQP5IoMu3MwsqRgbCCDlNyUv7qRmQo


In [1]:
import json
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

def convert_to_forms_api_requests(data):
    """Parses the JSON list and converts it into Google Forms API createItem requests."""
    requests = []
    index_counter = 0
    current_section = None

    for item in data:
        # Skip the title configuration (handled in the main function)
        if item.get("id") == 0:
            continue
            
        section_id = item.get("section_id")
        
        # Detect section changes and insert a page break
        if section_id and section_id != current_section:
            if current_section is not None:
                requests.append({
                    "createItem": {
                        "item": {
                            "title": f"Section: {section_id}",
                            "pageBreakItem": {}
                        },
                        "location": {"index": index_counter}
                    }
                })
                index_counter += 1
            current_section = section_id

        # Setup the basic item structure
        question_type = item.get("type")
        item_title = item.get("question")
        options = item.get("options", [])
        
        item_config = {"title": item_title}
        question_config = {"required": True}
        
        # Map your custom JSON types to Google Forms API types
        if question_type == "single_choice":
            question_config["choiceQuestion"] = {
                "type": "RADIO",
                "options": [{"value": opt} for opt in options]
            }
        elif question_type == "multiple_choice":
            question_config["choiceQuestion"] = {
                "type": "CHECKBOX",
                "options": [{"value": opt} for opt in options]
            }
        elif question_type == "text":
            question_config["textQuestion"] = {
                "paragraph": True
            }
            
        item_config["questionItem"] = {"question": question_config}
        
        # Add the constructed question to the batch requests
        requests.append({
            "createItem": {
                "item": item_config,
                "location": {"index": index_counter}
            }
        })
        index_counter += 1

    return {"requests": requests}

def mcp_survey_executor(questions_data: list) -> str:
    configs = json.load(open('./configs.json'))["GOOGLE_KEYS"]
    creds = None
    
    # 1. AUTHENTICATION
    if os.path.exists(configs["Token_json"]):
        creds = Credentials.from_authorized_user_file(configs["Token_json"], configs['SCOPES'])
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                configs["GOOGLE_APPLICATION_CREDENTIALS"], configs['SCOPES']
            )
            creds = flow.run_local_server(port=0) 
        with open(configs["Token_json"], "w") as token:
            token.write(creds.to_json())
    
    # 2. BUILD SERVICE
    form_service = build("forms", "v1", credentials=creds)
    
    # 3. EXTRACT TITLE FROM JSON (ID: 0)
    survey_title = "Untitled Survey"
    for item in questions_data:
        if item.get("id") == 0:
            survey_title = item.get("survey_title", survey_title)
            break

    # 4. CREATE INITIAL FORM
    result = form_service.forms().create(body={
        "info": {
            "title": survey_title,
            "documentTitle": survey_title
        }
    }).execute()
    
    form_id = result["formId"]
    print(f"Form Created! ID: {form_id}")

    # 5. GENERATE AND APPLY QUESTIONS
    api_payload = convert_to_forms_api_requests(questions_data)
    
    if api_payload["requests"]:
        form_service.forms().batchUpdate(formId=form_id, body=api_payload).execute()

    # 6. VERIFY AND OUTPUT
    final_form = form_service.forms().get(formId=form_id).execute()
    print(f"\nSurvey successfully published: https://docs.google.com/forms/d/{form_id}/edit")
    return final_form

if __name__ == "__main__":
    # Ensure you are reading the JSON and passing the parsed list/dict, not the raw string
    with open('./results/final_survey.json', 'r', encoding='utf-8') as f:
        survey_data = json.load(f)
        
    mcp_survey_executor(survey_data)

Form Created! ID: 1FqOxi_RSukXOtwcmcfcaRazXnnBTE7V6pXXi1CT04jU

Survey successfully published: https://docs.google.com/forms/d/1FqOxi_RSukXOtwcmcfcaRazXnnBTE7V6pXXi1CT04jU/edit
